# CLIP loop

Fetch random images repeatedly and keep the closest and furthest CLIP matches to a text prompt.

In [ ]:
import torch
from diffusers.utils import load_image
from transformers import CLIPModel, CLIPProcessor

# Edit these inputs.
prompt = "a smiling person"
iterations = 20
image_url = "https://picsum.photos/seed/{i}/512"

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id).to(device).eval()
processor = CLIPProcessor.from_pretrained(model_id)

text_inputs = processor(text=[prompt], return_tensors="pt").to(device)
with torch.inference_mode():
    target = model.get_text_features(**text_inputs)
    target /= target.norm(dim=-1, keepdim=True)

best = (-float("inf"), None)
worst = (float("inf"), None)

for i in range(iterations):
    image = load_image(image_url.format(i=i)).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.inference_mode():
        features = model.get_image_features(**inputs)
        features /= features.norm(dim=-1, keepdim=True)
        score = float(features @ target.T)
    if score > best[0]:
        best = (score, image.copy())
    if score < worst[0]:
        worst = (score, image.copy())
    print(f"{i + 1:02d}: {score:.4f}")

display(best[1], worst[1])
print(f"best={best[0]:.4f}, worst={worst[0]:.4f}")